<!-- Assignment 3 - SS 2024 -->

# Monitoring and advanced CNNs  (20 points)

This notebook contains one of the assignments for the exercises in Deep Learning and Neural Nets 2.
It provides a skeleton, i.e. code with gaps, that will be filled out by you in different exercises.
All exercise descriptions are visually annotated by a vertical bar on the left and some extra indentation,
unless you already messed with your jupyter notebook configuration.
Any questions that are not part of the exercise statement do not need to be answered,
but should rather be interpreted as triggers to guide your thought process.

**Note**: The cells in the introductory part (before the first subtitle)
perform all necessary imports and provide utility functions that should work without (too much) problems.
Please, do not alter this code or add extra import statements in your submission, unless explicitly allowed!

<span style="color:#d95c4c">**IMPORTANT:**</span> Please, change the name of your submission file so that it contains your student ID!

In this assignment, the main goal is to get familiar with neural network hyperparameter search.
More specifically, you will perform hyperparameter search on some real-world data.
To prepare you for the search, we will first look at how you can monitor the training progress.

In [10]:
import random
from pathlib import Path

import torch
import torchvision
from torch import nn, optim
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, random_split

torch.manual_seed(1806)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [11]:
# google colab data management
import os.path

try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    _home = 'gdrive/MyDrive/'
except ImportError:
    _home = '~'
finally:
    data_root = os.path.join(_home, '.pytorch')

print(data_root)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
gdrive/MyDrive/.pytorch


## Monitoring

Training a deep neural network with millions of parameters can cost quite some time. To make sure the network is training as expected,
it is crucial to monitor training progress in real time.

As a matter of fact, the `update` and `evaluate` functions
already implement some sort of ad hoc monitoring by providing the list of errors in a batch.
This list can be used to print the mean loss after every epoch
and can therefore be used to get an idea of how learning is progressing.
This specific implementation of monitoring the loss is not very flexible, however,
since it is not possible to access the information before the epoch has finished. Therefore, it is often necessary to log information more frequently. Moreover, additional information that show the model's training dynamics or help debugging a flawed model is frequently logged.

To thid end, various libraries exist. One of the most popular tools is [Weights & Biases (wandb)](https://wandb.ai/), which has largely replaced [Tensorboard](https://www.tensorflow.org/tensorboard) in recent years.

The standard pattern in most ML projects is straightforward:

1. **`wandb.init()`** — start a new run (with project name, config, etc.)
2. **`wandb.log()`** — log metrics (loss, accuracy, ...) during training
3. **`wandb.finish()`** — close the run when done

In this section, we integrate wandb directly into a `Trainer` class
so that every training run is automatically tracked.

The `Trainer` class provides the same functionality as the list that
you might have used in the current `update` and `evaluate` functions.
However, it also makes it possible to extend the functionality
of both functions without the need to interfere with existing code.

Note that there are libraries and frameworks out there that provide
(parts of) the functionality we will implement in what follows.
Two example frameworks that directly build on pytorch are
[pytorch-lightning](https://www.pytorchlightning.ai/)
and [pytorch ignite](https://pytorch.org/ignite/).

### Exercise 1: Trainer with Monitoring (3 points)

Below is a `Trainer` class that organises the training loop.
The goal of this exercise is to integrate wandb logging directly into the Trainer,
as is common practice in real ML projects.

 > Complete the `Trainer` class so that it:
 > - Initialises a wandb run in `__init__` (using the provided `wandb_config`).
 > - Logs the **per-batch loss** during `update()` as `"train/batch_loss"` at every gradient step.
 > - Logs the **average training and validation loss** after each epoch as `"train/loss"` and `"valid/loss"`.
 > - Calls `wandb.finish()` at the end of `train()`.
 >
 > The `train()` method should return a dict `{"train": ..., "valid": ...}` with the final average losses.

**Note:** Use `wandb.init(mode="offline")` in the config if you don't want to log to the cloud during development.

In [12]:
# install wandb, if this throws an error
import wandb

In [13]:
class Trainer:
    """ Class to organise learning and monitoring. """

    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        optimiser: optim.Optimizer,
        wandb_config: dict = None,
    ):
        """
        Parameters
        ----------
        model : torch.nn.Module
            Neural Network that will be trained.
        criterion : torch.nn.Module
            Loss function to use for training.
        optimiser : torch.optim.Optimizer
            Optimiser for training.
        wandb_config : dict, optional
            Configuration dict passed to wandb.init().
            Useful keys: project, name, config, mode, ...
        """
        self.model = model
        self.criterion = criterion
        self.optimiser = optimiser

        self.epoch = 0
        self.global_step = 0

        # Initialize wandb
        # YOUR CODE HERE

        # if no config is handed over, use an empty dictionary
        # safely call wandb.init
        if wandb_config is None:
            wandb_config = {}

        # starts a new weights & biases run
        # run later saves all training metrics
        self.run = wandb.init(**wandb_config)

    def state_dict(self):
        """ Current state of learning. """
        return {
            "model": self.model.state_dict(),
            "objective": self.criterion.state_dict(),
            "optimiser": self.optimiser.state_dict(),
            "num_epochs": self.epoch,
            "num_updates": self.global_step,
        }

    @property
    def device(self):
        """ Device of the (first) model parameters. """
        return next(self.model.parameters()).device

    @torch.no_grad()
    def evaluate(self, batches: DataLoader):
        """
        One epoch of evaluating the network.

        Parameters
        ----------
        batches : DataLoader
            An iterator over mini-batches of data to use for updating.
        tag : str, optional
            Identification tag for tracking loss values.

        Returns
        -------
        avg_loss : float
            The average loss over all mini-batches.
        """
        self.model.eval()
        device = self.device

        losses = []
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            losses.append(loss.item())

        avg_loss = sum(losses) / len(losses)
        return avg_loss

    @torch.enable_grad()
    def update(self, batches: DataLoader):
        """
        One epoch of updating the network.

        Parameters
        ----------
        batches : DataLoader
            An iterator over mini-batches of data to use for updating.
        tag : str, optional
            Identification tag for tracking loss values.

        Returns
        -------
        avg_loss : float
            The average loss over all mini-batches.
        """
        self.model.train()
        device = self.device

        losses = []
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            losses.append(loss.item())

            self.optimiser.zero_grad()
            loss.backward()
            self.optimiser.step()

            # use wandb for logging train statistics during an epoch
            # YOUR CODE HERE

            # increase gradient step counter
            self.global_step += 1

            # log batch loss after every optimizer step
            # step=self.global_step ensures that the x-axis is correct
            wandb.log(
                {"train/batch_loss": loss.item()},
                step=self.global_step
            )

        avg_loss = sum(losses) / len(losses)
        return avg_loss

    def train(self, train_batches, valid_batches=None, num_epochs: int = 1):
        """
        Train the network for multiple epochs.

        Parameters
        ----------
        train_batches : DataLoader
            The training data for updating the network.
        valid_batches : DataLoader, optional
            The validation data for estimating the generalisation performance.
        num_epochs : int, optional
            The number of epochs to train.

        Returns
        -------
        results : dict
            The average loss estimates after `num_epochs` epochs.

        """
        if valid_batches is None:
            valid_batches = ()

        # implement the whole training loop
        # log metrics
        # YOUR CODE HERE

        # training loop over several epochs
        for _ in range(num_epochs):
            self.epoch += 1

            # training per epoch
            train_loss = self.update(train_batches)

            # validate only if validation data is available
            if valid_batches:
                valid_loss = self.evaluate(valid_batches)

                # log training and validation loss after every epoch
                wandb.log(
                    {
                        "train/loss": train_loss,
                        "valid/loss": valid_loss,
                    },
                    step=self.global_step
                )
            else:
                # if no valition data is available, only log training loss
                wandb.log(
                    {"train/loss": train_loss},
                    step=self.global_step
                )
                valid_loss = None

        # end the run
        wandb.finish()

        return {"train": train_loss, "valid": valid_loss}

In [14]:
# Sanity check
# Build a tiny dataset and loader for testing
from torchvision import transforms

_dummy_data = torch.randn(128, 1, 8, 8)
_dummy_labels = torch.randint(0, 3, (128,))
_dummy_ds = torch.utils.data.TensorDataset(_dummy_data, _dummy_labels)
loader = DataLoader(_dummy_ds, batch_size=32)

conv_net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 3),
)


In [15]:
# Sanity check
# basic training loop
trainer = Trainer(
    model=conv_net.to(device),
    criterion=nn.CrossEntropyLoss(),
    optimiser=optim.Adam(conv_net.parameters(), lr=1e-2),
    wandb_config={"project": "dl-test", "mode": "offline", "name": "sanity-check"},
)
results = trainer.train(loader, loader, num_epochs=3)

assert "train" in results, "Could not find training loss in results"
assert "valid" in results, "Could not find validation loss in results"
assert isinstance(results["train"], float), f"Expected float, got {type(results['train'])}"
assert isinstance(results["valid"], float), f"Expected float, got {type(results['valid'])}"
assert (trainer.epoch in (2, 3)), f"Expected 2 or 3 epochs, got {trainer.epoch}" #changed form (1,2) to (2,3)
assert trainer.global_step > 0, f"Global_step not updated"

train/batch_loss,█▆▆▇▃▃▃▅▁▂▁▃
train/loss,█▄▁
valid/loss,█▄▁
train/batch_loss,0.94592
train/loss,0.87687
valid/loss,0.81976


## Hyperparameter Search

Finding good hyperparameters for a model is a general problem in machine learning (or even statistics).
However, neural networks are (in)famous for their large number of hyperparameters.
To list a few: learning rate, batch size, epochs, pre-processing, layer count, neurons for each layer,
activation function, initialisation, normalisation, layer type, skip connections, regularisation, ...
Moreover, it is often not possible to theoretically justify a particular choice for a hyperparameter.
E.g. there is no way to tell whether $N$ or $N + 1$ neurons in a layer would be better, without trying it out.
Therefore, hyperparameter search for neural networks is an especially tricky problem to solve.

###### Manual Search

The most straightforward approach to finding good hyperparameters is to just
try out *reasonable* combinations of hyperparameters and pick the best model (using e.g. the validation set).
The first problem with this approach is that it requires a gut feeling as to what *reasonable* combinations are.
Moreover, it is often unclear how different hyperparameters interact with each other,
which can make an irrelevant hyperparameter look more important than it actually is or vice versa.
Finally, manual hyperparameter search is time consuming, since it is generally not possible to automate.

###### Grid Search

Getting a feeling for combinations of hyperparameters is often much harder than for individual hyperparameters.
The idea of grid search is to get a set of *reasonable* values for each hyperparameter individually
and organise these sets in a grid that represents all possible combinations of these values.
Each combinations of hyperparameters in the grid can then be run simultaneously,
assuming that so much hardware is available, which can speed up the search significantly.

###### Random Search

Since there are plenty of hyperparameters and each hyperparameters can have multiple *reasonable* values,
it is often not feasible to try out every possible combination in the grid.
On top of that, most of the models will be thrown away anyway because only the best model is of interest,
even though they might achieve similar performance.
The idea of random search is to randomly sample configurations, rather than choosing from pre-defined choices.
This can be interpreted as setting up an infinite grid and trying only a few --- rather than all --- possibilities.
Under the assumption that there are a lot of configurations with similarly good performance as the best model,
this should provide a model that performs very good with high probability for a fraction of the compute.

###### Bayesian Optimisation

Rather than picking configurations completely at random,
it is also possible to guide the random search.
This is essentially the premise of Bayesian optimisation:
sample inputs and evaluate the objective to find which parameters are likely to give good performance.

Bayesian optimisation uses a function approximator for the objective
and what is known as an *acquisition* function.
The function approximator, or *surrogate*,
has to be able to model a distribution over function values, e.g. a Gaussian Process.
The acquisition function then uses these distributions
to find where the largest improvements can be made, e.g. using the cdf.
For a more elaborate explanation of Bayesian optimisation,
see e.g. [this tutorial](https://arxiv.org/abs/1807.02811)

This approach is less parallellisable than grid or random search,
since it uses the information from previous runs to find good sampling regions.
However, often there are more configurations to be tried out than there are computing devices
and it is still possible to sample multiple configurations at each step with Bayesian Optimisation.
Also consider [this paper](https://papers.nips.cc/paper/4522-practical-bayesian-optimization-of-machine-learning-algorithms) in this regard.

###### Neural Architecture Search

Instead of using Bayesian optimisation,
the problem of hyperparameter search can also be tackled by other optimisation algorithms.
This approach is also known as *Neural Architecture Search* (NAS).
There are different optimisation strategies that can be used for NAS,
but the most common are evolutionary algorithms and (deep) reinforcement learning.
Consider reading [this survey](http://jmlr.org/papers/v20/18-598.html)
to get an overview of how NAS can be used to construct neural networks.

## Efficient CNNs

In recent times CNNs have become more computationally efficient. Traditional convolutional layers apply filters across the entire depth of the input volume, mixing all the input channels to produce a single output channel. Depthwise separable convolutions, introduced as a key innovation in architectures like Xception, are a more efficient variant of the standard convolution operation. This process is divided into two layers: the depthwise convolution and the pointwise convolution. In the depthwise convolution, a single filter is applied per input channel, which significantly reduces the computational cost. Following this, a 1x1 convolution (pointwise convolution) is applied to combine the outputs of the depthwise layer, creating a new set of feature maps. This approach drastically reduces the number of parameters and computations, making the network more efficient and faster, which is especially beneficial for mobile and embedded devices.

<img src="https://www.researchgate.net/publication/358585116/figure/fig1/AS:1127546112487425@1645839350616/Depthwise-separable-convolutions.png" />

Squeeze-and-Excitation layers introduce an additional level of adaptivity in CNNs, enabling the network to perform dynamic channel-wise feature recalibration. Squeeze-and-Exitation blocks are usually executed after a convolutional layer or block
and before the residual connection by a series of relatively inexpensive computations

1. A three dimensional input consisting of different channels and the two spati l
dimensions is compressed into one dimension by global aver ge pooling. As a res lt
the spatial information is squeezed into one descriptor per channel.
2. The squeezed data is transformed by a two layer feed-forward neural network.  fter
the first linear layer ReLU is used as activation functi n and after the se ond a
sigmoid function is applied. This normalizes the output between 0 and 1 and can be
interpreted as the significance per channel.
3. The result is used to scale the input of the Squeeze-and-Exitation block by an element-
wise multiplication.

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*bmObF5Tibc58iE9iOu327w.png" />



### Exercise 2: Create an efficient CNN (4 points)

Today, neural networks frequently have millions or billions of parameters. However, CNNs have become more computationally efficient over the years. How far can you get with a limited amount of compute?

> Create an efficient CNN with less than 30.000 parameters.
> Use at least one depthwise separable or groupwise convolution or apply at least one squeeze-and-exitation layer after a convolution.

Hint: Skip-connections and Normalization layers are frequently used to stabilize the training behavior of deep CNNs.

In [16]:
class EfficientCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        # TODO: implement __init__

        super().__init__()

        # The feature extractor uses standard and depthwise separable convolutions
        # to keep the number of parameters low.
        self.features = nn.Sequential(

            # First standard convolution:
            # maps the input image to 24 feature channels (davor 16)
            nn.Conv2d(in_channels, 24, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(24), # normalize features for more stable training
            nn.ReLU(inplace=True), # non-linearity

            # First depthwise separable convolution block:
            # depthwise convolution applies one filter per input channel
            nn.Conv2d(24, 24, kernel_size=3, padding=1, groups=24, bias=False),
            # pointwise convolution mixes channel information
            nn.Conv2d(24, 48, kernel_size=1, bias=False), #(davor 32)
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),

            # Downsample spatial resolution from 48x48 to 24x24
            nn.MaxPool2d(2),

            # Second depthwise separable convolution block
            nn.Conv2d(48, 48, kernel_size=3, padding=1, groups=48, bias=False),
            nn.Conv2d(48, 96, kernel_size=1, bias=False),
            nn.BatchNorm2d(96), #davor 64
            nn.ReLU(inplace=True),

            # Downsample spatial resolution from 24x24 to 12x12
            nn.MaxPool2d(2),

            # Third depthwise separable convolution block
            nn.Conv2d(96, 96, kernel_size=3, padding=1, groups=96, bias=False),
            nn.Conv2d(96, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), #(davor 96)
            nn.ReLU(inplace=True),

            # Global average pooling reduces each channel to a single value
            # Output shape becomes: (batch_size, 128, 1, 1)
            nn.AdaptiveAvgPool2d(1)
        )

        # Final classification layer:
        # takes the 128 pooled features and maps them to class scores
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # TODO: implement forward

        # Pass the input through the convolutional feature extractor
        x = self.features(x)

        # Flatten from shape (batch_size, 96, 1, 1) to shape (batch_size, 96)
        x = torch.flatten(x, 1)

        # Compute the final class logits
        x = self.classifier(x)

        return x

# YOUR CODE HERE
model = EfficientCNN(in_channels=3, num_classes=10)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Number of trainable parameters: {num_params}")

EfficientCNN(
  (features): Sequential(
    (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=24, bias=False)
    (4): Conv2d(24, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (5): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
    (9): Conv2d(48, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (10): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (13): Conv

In [21]:
# sanity-check
model = EfficientCNN(in_channels=3, num_classes=10).to(device)
model(torch.zeros((1, 3, 32, 32)).to(device))
print("number of parameters: ", sum([p.numel() for p in model.parameters()]))

number of parameters:  22090


In [22]:
# Test Cell: do not edit or delete!

In [23]:
# Test Cell: do not edit or delete!

### Exercise 3: Training (4 points)

In order to get a feeling for hyperparameter search, you have to try it out on some example. You can use the monitoring tools from previous exercises to log performance and get a feeling for which hyperparameters work well.

> Train your EfficientCNN on CIFAR10 using the Trainer class. Use hyperparameter search for the learning rate, optimizer and maybe even the model architecture to get a CrossEntropyLoss < 1.5 within 10 epochs of training with a fixed batch size of 1024.

In [24]:
# TODO: Cell for Hyperparameter search, you can freely edit or delete this code
train_dataset = torchvision.datasets.CIFAR10(data_root, train=True, transform=transforms.ToTensor(), download=True)
test_dataset = torchvision.datasets.CIFAR10(data_root, train=False, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=True, num_workers=2)
model = EfficientCNN(in_channels=3, num_classes=10)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
trainer = Trainer(model,
                  criterion,
                  optimizer)
trainer.train(train_loader, test_loader, num_epochs=10)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


train/batch_loss,█▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▄▃▄▃▃▃▂▃▂▃▂▂▂▂▁▁▁▁
train/loss,█▇▆▅▄▃▂▂▁▁
valid/loss,█▇▆▅▄▃▂▂▁▁
train/batch_loss,1.84844
train/loss,1.86035
valid/loss,1.84613


{'train': 1.8603481443560854, 'valid': 1.846126341819763}

In [25]:
# Test Cell: do not edit or delete!

## Beyond Convolutions: Alternative Vision Architectures

Convolutional neural networks have dominated computer vision for over a decade.
However, several alternative architectures have emerged that challenge
the assumption that convolutions are necessary for visual understanding.

###### Vision Transformers (ViT)

The [Vision Transformer (ViT)](https://arxiv.org/abs/2010.11929) applies the Transformer architecture,
originally developed for NLP, directly to image classification.
An image is split into fixed-size patches (e.g. 16×16), each patch is linearly embedded,
and the resulting sequence of patch embeddings is processed by a standard Transformer encoder.
A special `[CLS]` token is prepended to the sequence and its final representation is used for classification.
ViTs achieve state-of-the-art results but typically require very large datasets
or extensive pre-training to outperform CNNs.

###### MLP-Mixer

The [MLP-Mixer](https://arxiv.org/abs/2105.01601) takes a more radical approach:
it uses **only MLPs** — no convolutions, no self-attention.
Like ViT, the image is divided into non-overlapping patches that are linearly projected.
The architecture then alternates between two types of MLP layers:

1. **Token-mixing MLPs** — applied across the spatial (patch) dimension,
   allowing communication between different spatial locations.
   All channels of one patch are mixed with all channels of every other patch.
2. **Channel-mixing MLPs** — applied independently to each patch,
   mixing information across the feature/channel dimension.

Each mixer layer applies layer normalisation, followed by the MLP, and uses a residual connection:

$$U = X + W_2 \, \sigma(W_1 \, \text{LayerNorm}(X)^T)^T \quad \text{(token-mixing)}$$
$$Y = U + W_4 \, \sigma(W_3 \, \text{LayerNorm}(U)) \quad \text{(channel-mixing)}$$

Given the lack of inductive bias for image processing, this design is surprisingly competitive.

### Exercise 5: Implement an MLP-Mixer (4 points)

Now it is your turn to implement a vision architecture that uses **no convolutions and no attention**.

 > Implement an `MLPMixer` model for CIFAR-10 (32×32 RGB images, 10 classes) with **fewer than 30,000 parameters**.
 >
 > Your implementation must include:
 > - A **patch embedding** layer that splits the image into non-overlapping patches and projects them.
 > - At least **2 mixer layers**, each consisting of a token-mixing MLP and a channel-mixing MLP.
 > - **Residual connections** and **layer normalisation** in each mixer layer.
 > - A **global average pooling** + linear classification head.

**Hint:** With 32×32 images, a patch size of 8 gives you 16 patches.
Choose the hidden dimension and MLP expansion factors carefully to stay under the parameter budget.
GELU is the standard activation for mixer MLPs.


In [32]:
class MixerBlock(nn.Module):
    """A single Mixer layer: token-mixing followed by channel-mixing."""

    def __init__(self, num_patches, hidden_dim, token_mlp_dim, channel_mlp_dim):
        super().__init__()
        # Token-mixing: operates across the patch (spatial) dimension
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_mlp_dim),
            nn.GELU(),
            nn.Linear(token_mlp_dim, num_patches),
        )
        # Channel-mixing: operates across the channel/feature dimension
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(hidden_dim, channel_mlp_dim),
            nn.GELU(),
            nn.Linear(channel_mlp_dim, hidden_dim),
        )

    def forward(self, x):
        # x shape: (batch, num_patches, hidden_dim)
        # TODO: implement forward

        # Token-mixing:
        # First normalize across the feature dimension
        y = self.norm1(x)

        # Transpose so that the MLP can mix information across patches
        # Shape: (batch, hidden_dim, num_patches)
        y = y.transpose(1, 2)

        # Apply token-mixing MLP across the patch dimension
        y = self.token_mlp(y)

        # Transpose back to the original layout: (batch, num_patches, hidden_dim)
        y = y.transpose(1, 2)

        # Residual connection
        x = x + y

        # Channel-mixing:
        # Normalize again before mixing channels/features
        y = self.norm2(x)

        # Apply channel-mixing MLP across the hidden dimension
        y = self.channel_mlp(y)

        # Residual connection
        x = x + y

        return x


class MLPMixer(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, image_size=32, patch_size=8,
                 hidden_dim=32, num_layers=2, token_mlp_dim=64, channel_mlp_dim=64):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.patch_size = patch_size
        num_patches = (image_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size  # flattened patch

        # Patch embedding: project each flattened patch to hidden_dim
        self.patch_embed = nn.Linear(patch_dim, hidden_dim)

        # Mixer layers
        self.mixer_layers = nn.Sequential(*[
            MixerBlock(num_patches, hidden_dim, token_mlp_dim, channel_mlp_dim)
            for _ in range(num_layers)
        ])

        # Classification head
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # TODO: implement forward

        # x has shape: (batch, channels, height, width)
        B, C, H, W = x.shape
        P = self.patch_size

        # Split the image into non-overlapping patches
        # After unfold:
        # shape -> (batch, channels, H/P, W/P, P, P)
        x = x.unfold(2, P, P).unfold(3, P, P)

        # Rearrange patches so that each patch becomes one token
        # Shape -> (batch, num_patches, channels, P, P)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()

        # Flatten each patch
        # Shape -> (batch, num_patches, patch_dim)
        x = x.view(B, -1, C * P * P)

        # Project flattened patches to the hidden dimension
        # Shape -> (batch, num_patches, hidden_dim)
        x = self.patch_embed(x)

        # Pass tokens through the Mixer blocks
        x = self.mixer_layers(x)
        #print(x)
        #print(x.shape)

        # Final normalization before classification
        x = self.norm(x)

        # Global average pooling over all patches
        # Shape -> (batch, hidden_dim)
        x = x.mean(dim=1)

        # Linear classification head
        # Shape -> (batch, num_classes)
        x = self.head(x)

        return x


# YOUR CODE HERE


In [36]:
# sanity-check
mixer = MLPMixer(in_channels=3, num_classes=10, image_size=32, patch_size=8,
                 hidden_dim=32, num_layers=2, token_mlp_dim=64, channel_mlp_dim=64).to(device)
out = mixer(torch.zeros((1, 3, 32, 32)).to(device))
print(f"Output shape: {out.shape}")
num_params = sum(p.numel() for p in mixer.parameters())
print(f"Number of parameters: {num_params}")


Output shape: torch.Size([1, 10])
Number of parameters: 19466


In [ ]:
# Test Cell: do not edit or delete!


In [ ]:
# Test Cell: do not edit or delete!


### Exercise 5: Train the MLP-Mixer (4 points)

 > Train your `MLPMixer` on CIFAR-10 using the `Trainer` class.
 > Use wandb to log and compare runs.
 > Achieve a `CrossEntropyLoss < 1.5` within 10 epochs.

**Hint:** MLP-Mixers can be sensitive to the learning rate.
Try Adam or AdamW with learning rates in the range 1e-3 to 5e-3.

In [35]:
# TODO: Train your MLPMixer and tune hyperparameters
# YOUR CODE HERE


# Configure wandb for local/offline experiment tracking
wandb_config = {
    "project": "mlp-mixer-cifar10",
    "name": "mixer-adamw-lr3e-3",
    "mode": "offline",   # prevents login prompts and cloud syncing
    "config": {
        "model": "MLPMixer",
        "dataset": "CIFAR10",
        "optimizer": "AdamW",
        "learning_rate": 3e-3,
        "batch_size": 128,
        "num_epochs": 10,
        "patch_size": 8,
        "hidden_dim": 32,
        "num_layers": 2,
        "token_mlp_dim": 64,
        "channel_mlp_dim": 64,
    }
}

# Create MLP-Mixer and move it to the selected device
model = MLPMixer(
    in_channels=3,
    num_classes=10,
    image_size=32,
    patch_size=8,
    hidden_dim=32,
    num_layers=2,
    token_mlp_dim=64,
    channel_mlp_dim=64,
).to(device)

# CrossEntropyLoss = standard choice for multi-class classification
criterion = nn.CrossEntropyLoss()

# AdamW = good default choice for Mixer-like architectures
optimiser = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)

# Create trainer with wandb logging enabled
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimiser=optimiser,
    wandb_config=wandb_config,
)

# Train for 10 epochs and also evaluate on the validation set
results = trainer.train(train_loader, test_loader, num_epochs=10)

# Print results
print(results)

train/batch_loss,███▇▇▆▇▆▆▅▅▅▄▄▅▃▃▃▃▃▃▃▃▃▂▂▃▂▂▂▂▂▂▂▂▂▂▁▁▂
train/loss,█▅▃▂▁▁
valid/loss,█▅▃▂▂▁
train/batch_loss,1.33758
train/loss,1.34902
valid/loss,1.36909


train/batch_loss,█▆▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▃▂▂▃▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▂▁▁
train/loss,█▅▄▃▃▂▂▂▁▁
valid/loss,█▅▄▃▃▂▂▂▁▁
train/batch_loss,1.1964
train/loss,1.25792
valid/loss,1.30183


{'train': 1.2579186619544516, 'valid': 1.3018286108970643}


In [ ]:
# Test Cell: do not edit or delete!


### Conclusions (1 point)
The questions below should help you to reflect on the experiments. Enter your answer directly in this cell or create a new cell.
 - If you compare CNNs and MLP-mixer: What is the more universal architecture? Which inductive biases do they have?
 - You might have noticed that the difference of training and test error is not the same in the efficient CNN as in the MLP-mixer. What could be the reason? Which architecture would you use for image classification?

 - Answers:

- MLP-Mixer is more universal, because it is not specialized for images.
- CNNs have stronger inductive biases: locality and translation equivariance (assumes nearby pixels belong together)
- Therefore: MLP-Mixers have weaker image-specific biases.

- The different train/test gap is likely due to generalization.
- CNNs usually generalize better on images, especially with smaller datasets.
- MLP-Mixers often need more data and tuning (fewer built in assumptions)
- For image classification, I would usually choose a CNN (easier to train).